# [1.3.1] Linear probes ⑤ — 논문 초안: 증거를 주장으로 바꾸기

1부의 마지막 단계다. 저자는 ①~④의 결과를 하나의 논증으로 묶는다. 이 notebook의 일은 두 가지다.

1. **주장–증거 지도:** 초록의 문장 하나하나가 정확히 어느 실험의 어느 숫자에 기대는지 연결한다.
2. **범위 맞추기:** 각 문장의 범위(이 모델, 이 층, 이 데이터, 이 prompt 형식)를 증거가 허락하는 만큼만 쓴다.

새 실험은 없다. `results/`에 저장된 ①~④의 숫자를 모아 읽는다. 다시 실행해 숫자가 바뀌면 아래 표도 따라 바뀐다.

## 학습 목표, 필수 경로, stop rule

1. "읽힘 → 선택성 → 일반화 → 인과"의 **증거 사다리**에서 각 단계가 어떤 대안 설명을 약화시키는지 말한다.
2. 초안 초록의 문장마다 근거 실험과 범위를 표시한다.
3. 저자가 스스로 쓰는 한계(threats to validity)와 심사위원이 찌를 곳을 구분한다.

### ⑥으로 넘어가도 되는 기준

- 아래 초록에서 **증거보다 넓게 쓴 단어**를 세 개 이상 찾을 수 있다.
- "이 방향은 진실을 표현한다"와 "이 방향은 우리가 시험한 긍정문 데이터에서 참/거짓과 상관되고, 밀면 판단이 바뀐다"의 차이를 설명할 수 있다.

## 공통 setup

#### Cell 01 · 실행 환경과 공통 import

- **이번 셀의 질문:** 결과 파일을 읽을 수 있는가?
- **출력에서 볼 것:** 버전 출력과 함께 `results/` 안의 파일 목록을 본다. 01~04가 모두 있어야 한다.
- **해석 범위:** 이 notebook은 GPU를 쓰지 않는다.
- **다음 단계의 목적:** ①~④의 핵심 수치를 하나의 표로 모은다.

In [1]:
# 공통 setup — import는 이 셀 한 곳에서만 한다.
from __future__ import annotations

import gc
import sys
from importlib.metadata import version
from pathlib import Path

HERE = Path.cwd()
if not (HERE / "probe_lab.py").exists():  # 저장소 root 등 다른 위치에서 연 경우
    HERE = next(
        p
        for p in [HERE / "chapter1_transformer_interp/exercises/part31_linear_probes_ko", *HERE.parents]
        if (p / "probe_lab.py").exists()
    )
sys.path.insert(0, str(HERE))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

import probe_lab as pl
import tests

pl.use_style()
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)

print({
    "torch": torch.__version__,
    "transformers": version("transformers"),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "model": pl.DEFAULT_MODEL,
})

print(sorted(p.name for p in pl.RESULTS_DIR.glob("*.json")))
R = {name: pl.load_results(name) for name in ["01_observation", "02_controls", "03_generalization", "04_causal"]}

{'torch': '2.11.0+cu128', 'transformers': '5.18.0', 'gpu': 'NVIDIA RTX A6000', 'model': 'meta-llama/Llama-3.1-8B-Instruct'}
['01_observation.json', '02_controls.json', '03_generalization.json', '04_causal.json', '05_draft.json', '06_polarity.json', '07_use.json', '08_generations.json', '08_monitor.json']


# 1️⃣ 증거 사다리

1부에서 쌓은 증거를 아래에서 위로 놓으면 사다리가 된다. 위 단계는 아래 단계를 전제로 한다. 각 단계가 **새로** 약화시키는 대안 설명이 있다.

| 단계 | 묻는 것 | 새로 약화시키는 대안 |
|---|---|---|
| 1. 읽힘 | 학습에 없던 문장에서 선형으로 맞히는가? | A1 누수, A2 선택 편향 |
| 2. 선택성 | 표면 통계·probe 용량으로 설명되는가? | A3 표면 통계, A4 용량, (A5 일부) |
| 3. 일반화 | 다른 데이터에서도 같은 방향인가? | A6 데이터셋 특수성 (긍정문 안에서) |
| 4. 인과(충분성) | 그 방향으로 밀면 판단이 바뀌는가? 무작위 방향과 다른가? | A7 부수현상 (일부) |
| 5. 인과(필요성) | 그 정보가 없으면 판단하지 못하는가? | A7 (나머지) — **미검증** |
| 6. 실전 견고성 | 분포 이동·적대적 압력에서도 버티는가? | monitor로서의 유용성 — **미검증** |

#### Cell 02 · 결과 파일에서 증거 표 만들기

- **이번 셀의 질문:** 각 단계의 핵심 숫자는 무엇이고, 어느 notebook에서 왔는가?
- **출력에서 볼 것:** 표의 '핵심 수치' 열이 저장된 결과에서 자동으로 채워진다. 예: test AUROC 1.000, LR 귀무분포 평균 0.50, 역할극 거짓말 0.98, α=0.5에서 84%/99% vs 무작위 1%/0.2%.
- **해석 범위:** 표는 숫자를 모은 것이지 새 증거가 아니다. 숫자가 같아도 어떤 단어로 주장하는지는 저자의 선택이다. 그 선택이 다음 절의 주제다.
- **다음 단계의 목적:** 이 표를 근거로 초록 초안을 쓴다.

In [2]:
r1, r2, r3, r4 = R["01_observation"], R["02_controls"], R["03_generalization"], R["04_causal"]
test = {row["probe"]: row for row in r1["test"]}
null = {row["probe"]: row for row in r2["null"]}
surf = {row["features"]: row["test AUROC"] for row in r2["surface_baselines"]}
transfer = {row["dataset"]: row for row in r3["transfer"]}
dose = pd.DataFrame(r4["dose_response"])
d05 = dose[dose.alpha == 0.5].set_index("direction")
topics = ["sp_en_trans", "inventors", "animal_class", "element_symb", "facts"]

ladder = pd.DataFrame([
    {"단계": "1. 읽힘", "notebook": "①",
     "핵심 수치": f"L{r1['probe_layer']} test AUROC MM {test['MM']['test AUROC']:.3f}, LR {test['LR']['test AUROC']:.3f} (149개 미지 도시)"},
    {"단계": "2. 선택성", "notebook": "②",
     "핵심 수치": f"표면 baseline ≤ {max(v for k, v in surf.items() if 'layer' not in k):.3f}; random-init ≤ {r2['random_init_max_val_auroc']:.3f}; "
                 f"LR 귀무분포 {null['LR (100 perms)']['mean']:.2f}±{null['LR (100 perms)']['sd']:.3f}, p={null['LR (100 perms)']['p-value']:.3f}"},
    {"단계": "(기술) 최대 분산 축", "notebook": "②",
     "핵심 수치": f"PC1 분산 {r2['pc1_explained']:.0%}; MM 귀무분포 P(≥0.99)={null['MM (200 perms)']['P(null ≥ 0.99)']:.2f}, p={null['MM (200 perms)']['p-value']:.2f}"},
    {"단계": "3. 일반화", "notebook": "③",
     "핵심 수치": f"주제 5개 AUROC {min(transfer[t]['MM AUROC'] for t in topics):.2f}–{max(transfer[t]['MM AUROC'] for t in topics):.2f}; "
                 f"수 비교 {transfer['larger_than']['MM AUROC']:.2f}/{transfer['smaller_than']['MM AUROC']:.2f}; 독일어 {transfer['cities_de']['MM AUROC']:.2f}; "
                 f"역할극 거짓말 {r3['roleplay_mm_auroc']:.2f}"},
    {"단계": "4. 인과(충분성)", "notebook": "④",
     "핵심 수치": f"α=0.5: MM 뒤집힘 F→T {d05.loc['MM', 'flip F→T']:.0%}, T→F {d05.loc['MM', 'flip T→F']:.0%}; "
                 f"무작위 {d05.loc['random (mean of 20)', 'flip F→T']:.0%}, {d05.loc['random (mean of 20)', 'flip T→F']:.1%}"},
    {"단계": "5. 인과(필요성)", "notebook": "—", "핵심 수치": "미검증 (⑦)"},
    {"단계": "6. 실전 견고성", "notebook": "—", "핵심 수치": "미검증 (⑧)"},
])
pd.set_option("display.max_colwidth", 200)
display(ladder)

,단계,notebook,핵심 수치
0,1. 읽힘,①,"L12 test AUROC MM 1.000, LR 1.000 (149개 미지 도시)"
1,2. 선택성,②,"표면 baseline ≤ 0.475; random-init ≤ 0.578; LR 귀무분포 0.50±0.044, p=0.010"
2,(기술) 최대 분산 축,②,"PC1 분산 56%; MM 귀무분포 P(≥0.99)=0.30, p=0.10"
3,3. 일반화,③,주제 5개 AUROC 0.94–1.00; 수 비교 1.00/1.00; 독일어 1.00; 역할극 거짓말 0.98
4,4. 인과(충분성),④,"α=0.5: MM 뒤집힘 F→T 84%, T→F 99%; 무작위 1%, 0.2%"
5,5. 인과(필요성),—,미검증 (⑦)
6,6. 실전 견고성,—,미검증 (⑧)


# 2️⃣ 초록 초안

저자의 입장에서 쓴 초록이다. 대괄호의 `[S1]`… 표시는 문장 번호다. 영어로 쓴 것은 실제 논문이 그렇게 쓰이기 때문이다. 바로 아래에 문장별 근거와 범위를 붙였다.

> **Linear, Generalizing and Causal: A Truth Direction in Llama-3.1-8B-Instruct**
>
> [S1] Do language models internally represent whether a statement is true? [S2] Training linear probes on the residual stream of Llama-3.1-8B-Instruct, we find that the truth value of simple factual statements about held-out cities is linearly decodable at layer 12 with near-perfect accuracy (test AUROC 1.00). [S3] This is not explained by lexical baselines (AUROC ≤ 0.48), a randomly initialized model (≤ 0.58), or probe capacity (permutation test, p ≤ 0.01). [S4] Truth is in fact the dominant axis of variation: the first principal component alone explains 56% of the variance and separates true from false statements without labels. [S5] A direction learned only from city statements generalizes zero-shot to Spanish–English translations, chemical element symbols, numerical comparisons, German statements and real-world role-play lies (AUROC 0.81–1.00). [S6] Finally, this direction is causally implicated in the model's behavior: shifting activations along it flips the model's TRUE/FALSE judgments in both directions (84% and 99% of held-out statements), whereas random directions of equal norm do not (≤ 1%). [S7] Our results suggest that LLMs maintain a general, linearly represented truth variable that they use when evaluating statements, with implications for monitoring model honesty.

## 문장별 근거와 범위

| 문장 | 근거 | 증거가 허락하는 범위 | 넘친 부분 |
|---|---|---|---|
| S1 | (질문) | — | 질문 자체가 "represent ... true"라는 큰 개념을 깐다 |
| S2 | ① test AUROC 1.000 | Llama-3.1-8B-Instruct, layer 12, 마침표 위치, `The city of X is in Y.` 템플릿 | "simple factual statements"는 cities 하나에서 일반화한 표현 |
| S3 | ② baseline, random-init, LR 귀무분포 | **선형** 표면 baseline, random-init seed 1개, LR probe | MM probe로는 같은 검정이 p≈0.10이었다는 사실이 빠졌다 |
| S4 | ② PC1 56% | cities 데이터, layer 12 | 고정 템플릿 데이터라 참/거짓 외에 크게 변할 것이 없었을 가능성 |
| S5 | ③ zero-shot 전이 | **긍정문**, 이 평가 묶음 | 부정문을 시험하지 않았다. threshold는 전이되지 않았다(수 비교 accuracy 0.50) |
| S6 | ④ 용량–반응 | α=0.5, layer 8–12, 문장 끝 두 위치, 하나의 few-shot 형식 | α=1(GoT 설정)에서는 무작위 방향도 F→T 80%. 필요성은 미검증 |
| S7 | S2–S6의 종합 | — | "general", "truth variable", "use", "monitoring honesty" — 모두 증거보다 넓다 |

#### Cell 03 · ✍️ 직접 고쳐 쓰기

- **이번 셀의 질문:** S7을 증거가 허락하는 범위로 다시 쓰면 어떻게 되는가?
- **출력에서 볼 것:** 빈 셀이다. 위 표의 '범위' 열만 써서 S7을 한두 문장으로 다시 쓴다.
- **해석 범위:** 정답은 하나가 아니다. 다만 '진실', '일반적', '사용한다', '정직성 감시' 각각을 쓰려면 어떤 증거가 더 필요한지 적어 본다. ⑧에서 2부의 결과를 반영해 한 번 더 고쳐 쓴다.
- **다음 단계의 목적:** 저자가 스스로 쓰는 한계 절을 본다.

In [3]:
my_S7 = """
(여기에 다시 쓴 S7)
"""

# 3️⃣ 저자가 스스로 쓰는 한계 (Threats to validity)

좋은 저자는 심사위원보다 먼저 약점을 쓴다. 1부의 실험만으로 쓸 수 있는 한계 절이다.

1. **모델 하나, 층 하나.** layer 12는 cities validation의 분리도로 골랐다. 다른 모델·크기에서 같은 층 위치가 성립한다는 근거는 없다.
2. **템플릿 데이터, 긍정문만.** 일반화 평가에 부정문, 맥락 속 문장, 모델 자신이 생성한 문장이 없다.
3. **행동과 내부 상태가 일치하는 경우만 보았다.** 모델은 cities를 100% 맞힌다. probe가 행동이 모르는 것을 알려 주는 경우(그것이 감시 도구의 존재 이유다)는 시험하지 않았다.
4. **개입은 분포 밖일 수 있다.** α=0.5에서도 누적 개입 크기(약 11)가 원래 residual norm(약 8)보다 크다. 특이성은 α ≤ 0.5에서만 성립했다.
5. **충분성만 보았다.** "밀면 바뀐다"는 보였지만 "없으면 못 한다"는 보이지 않았다.
6. **threshold는 전이되지 않는다.** 순위는 옮겨 가도 판정 기준은 데이터마다 다시 맞춰야 했다.
7. **적대적 평가가 없다.** 모델이(또는 공격자가) probe를 피하려 할 때의 성능은 모른다.

# 4️⃣ 심사위원은 어디를 찌를까 — 2부 예고

| 심사위원의 질문 | 겨누는 문장 | 2부 위치 |
|---|---|---|
| "부정문에서도 같은 방향인가? 'Paris is **not** in France'는?" | S2, S5 ('truth') | ⑥ |
| "지금 읽은 것이 '참'인가, '도시와 국가가 짝이 맞음'인가?" | S2, S4 | ⑥ |
| "아무 개념이나 이렇게 잘 읽히지 않나? 예: 나라를 무작위로 두 묶음으로 나눈 '개념'" | S3 | ⑦ |
| "그 정보를 지우면 판단을 못 하나? 지우는 위치는 맞나?" | S6 (필요성) | ⑦ |
| "probe가 찾은 방향이 모델이 쓰는 특징인가, 개입이 다른 경로를 켠 것인가?" | S6 | ⑦ |
| "선형이 아닌 표현이라면 이 방법은 무엇을 놓치나?" | 방법론 | ⑦ |
| "거짓말 탐지기로 쓸 수 있나? 낮은 FPR, 분포 이동, 적대적 회피에서는?" | S7 ('monitoring') | ⑧ |

## 주장 장부 — 1부 최종

| ID | 주장 | 근거 | 아직 배제하지 못한 설명 | 상태 |
|---|---|---|---|---|
| C1 | L12에서 cities 참/거짓이 선형으로 분리된다. | ①② | — | 지지됨 |
| C2 | 표면 통계·probe 용량으로 설명되지 않는다. | ② | 비선형 표면 특징 | 지지됨 |
| D1 | 참/거짓이 L12 activation의 최대 분산 축이다. | ② | 템플릿 데이터의 성질일 가능성 | 관찰됨 |
| C3 | cities 방향은 여러 종류의 참/거짓 문장에 공통된 변수를 읽는다. | ③ | **부정문 미검증**, 공통 변수 ≠ 진실 가능성 | 지지됨 (긍정문 한정) |
| C4 | 그 방향은 모델의 TRUE/FALSE 판단에 인과적으로 관여한다. | ④ | 필요성, 분포 밖 개입, 착시 | 지지됨 (충분성) |
| C5 | (초록 S7) 모델은 일반적인 진실 변수를 가지고 그것을 쓴다. 정직성 감시에 쓸 수 있다. | C1–C4의 종합 | 위의 모든 것 | **주장됨 (증거 초과)** |

2부는 이 장부를 다시 연다. C5가 어디까지 깎이는지가 이 장의 결론이다.

#### Cell 04 · 결과 저장

- **이번 셀의 질문:** ⑧이 '전/후' 비교에 쓸 초안과 장부를 기록한다.
- **출력에서 볼 것:** `results/05_draft.json` 경로가 출력된다.
- **해석 범위:** 초안 문장과 장부 상태만 저장한다.
- **다음 단계의 목적:** ⑥ 반박 1 — 부정문과 극성.

In [4]:
ABSTRACT = {
    "S2": "truth value of held-out city statements is linearly decodable at layer 12 (test AUROC 1.00)",
    "S3": "not explained by lexical baselines, a randomly initialized model, or probe capacity",
    "S4": "truth is the dominant axis of variation (PC1 56%)",
    "S5": "generalizes zero-shot to other topics, numerical comparisons, German and role-play lies (AUROC 0.81-1.00)",
    "S6": "shifting along it flips TRUE/FALSE judgments both ways (84%/99%), random directions do not (<=1%)",
    "S7": "LLMs maintain a general, linearly represented truth variable that they use; implications for monitoring honesty",
}
pl.save_results("05_draft", {
    "abstract": ABSTRACT,
    "ladder": ladder.to_dict(orient="records"),
    "claims": [
        {"id": "C1", "status": "지지됨"}, {"id": "C2", "status": "지지됨"}, {"id": "D1", "status": "관찰됨"},
        {"id": "C3", "status": "지지됨 (긍정문 한정)"}, {"id": "C4", "status": "지지됨 (충분성)"},
        {"id": "C5", "status": "주장됨 (증거 초과)"},
    ],
})

PosixPath('/workspace/ARENA_3.0/chapter1_transformer_interp/exercises/part31_linear_probes_ko/results/05_draft.json')